# RNA / DCL4 complexes — energy + non-MoRF over-folding filter

**Kernel:** `abcfold-drbs-notebook` (`envs/notebook.yaml`).

Applies the two quality filters worked out in
`notebooks/drb2_drb4_domain_analysis.ipynb` to the **large RNA-containing
complexes**, where — as expected — the disordered-region over-folding is much
worse than in the DCL4-free binary complex:

1. **Garbage-energy filter.** Drop minimised structures whose final ChimeraX
   energy is a numerical-divergence outlier (robust median/MAD modified
   z-score, `|z| > 3.5`), and drop a whole backend if >50 % of its
   energy-assessed models are flagged. This removes RosettaFold3 in both
   complexes (final energies up to ~1e19 kJ/mol).

2. **Non-MoRF over-folding filter.** The disordered regions should carry
   essentially no stable secondary structure *except* at genuine
   fold-upon-binding motifs. This notebook defines those motifs by
   **ANCHOR2 peak detection** (the method developed at the end of
   `drb2_drb4_domain_analysis.ipynb`): run `scipy.signal.find_peaks` on the
   AIUpred ANCHOR2 (binding-induced-order) track and take each peak's
   prominence-defined span as a MoRF region — DRB2's near-saturated track is
   rolling-min **baseline-detrended** first so the crests can be resolved.
   DRB2/DRB4 sequences are identical across all three complexes (copied
   verbatim), so the `drb2_drb4` project's ANCHOR2 tracks apply directly by
   residue number. A model is flagged if, in the **non-MoRF** disordered
   residues, it forms `> MAX_PCT_HELIX_NONMORF` % helix **or** a single
   contiguous helical run of `>= MIN_LONG_HELIX_RUN` residues (DSSP).

**Disordered regions checked** (1-based, per-chain numbering)

| complex | region | chain | residues | MoRF exemption |
|---|---|---|---|---|
| both | DRB2 disordered | DRB2 | 189–434 | ANCHOR2 peak regions (detrended), ~132 res |
| both | DRB4 disordered | DRB4 | 151–291 | ANCHOR2 peak regions (raw track), ~67 res |
| `rna_ds_dcl4_drb2_drb4` only | DCL4 disordered linker core | DCL4 | 1586–1623 | *none — ANCHOR2 flat over the whole linker (max 0.41); the region is already trimmed to the AIUpred-disordered stretch, so 1529–1585 (predicted ordered) is not policed* |

The previous version of this notebook used two fixed MoRFchibi-consensus
windows (DRB2 425–434, DRB4 268–284); the "Reading" cells compare the
survivor counts under both definitions. The DCL4 region was also previously
the full structural linker 1529–1620 with no MoRF data; the 2026-09-02
DCL4 AIUpred/ANCHOR2 run
(`data/fold_inputs/rna_ds_dcl4_drb2_drb4/AIupred_output_dcl4.txt`) showed
1529–1585 is predicted ordered and ANCHOR2 carries no MoRF-strength peak
anywhere in the linker, so it is now trimmed to the disordered core.

**Prerequisite — `dssp_summary.csv` per complex** (generated with local
ChimeraX over every minimised `<fname>.pdb`):

```
chimerax --nogui --script "scripts/dssp_summary.py results/<complex>/dssp_manifest.txt results/<complex>/dssp_summary.csv"
```

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path

ROOT = Path("..")
TEMPLATE = "plotly_white"
BACKEND_PALETTE = px.colors.qualitative.Set2

# Figures / CSVs land under results/<complex>/figures/domain_analysis/<sub>.
# The canonical run uses FIG_SUBDIR; an alternate run (e.g. the
# "DCL4 over-folding filter disabled" variant near the bottom of this
# notebook) passes a `variant` suffix so its outputs go to a sibling folder
# (`energy_overfolding_filter_<variant>/`) and never clobber the canonical
# results.
FIG_SUBDIR = "energy_overfolding_filter"

def out_path(complex_name, filename, variant=""):
    sub = FIG_SUBDIR + (f"_{variant}" if variant else "")
    d = ROOT / "results" / complex_name / "figures" / "domain_analysis" / sub
    d.mkdir(parents=True, exist_ok=True)
    return d / filename

def save_fig(fig, complex_name, filename, variant=""):
    out = out_path(complex_name, filename, variant)
    fig.write_html(out, include_plotlyjs="cdn")
    print(f"Saved: {out}")
    fig.show()

## Configuration — complexes, disordered regions, thresholds

In [2]:
# --- filter thresholds (tune here) ---------------------------------------
MOD_Z_THRESHOLD        = 3.5    # |modified z-score| above this = garbage energy
WHOLE_BACKEND_DROP_FRAC = 0.5   # drop a backend if >this fraction of its assessed models are flagged
MAX_PCT_HELIX_NONMORF  = 20.0   # % helix in the non-MoRF disordered residues, above this = over-folded
MIN_LONG_HELIX_RUN     = 12     # a single contiguous helical run >= this (residues) = over-folded

# --- MoRF motifs via ANCHOR2 peak detection --------------------------------
# (method from drb2_drb4_domain_analysis.ipynb's "ANCHOR2 peak-based MoRF"
# section). DRB2/DRB4 sequences are identical across all three complexes, so
# the drb2_drb4 project's AIUpred tracks apply directly by residue number.
from scipy.signal import find_peaks, peak_widths

_AIU_DIR = ROOT / "data" / "fold_inputs" / "drb2_drb4"
PEAK_CFG = {
    "DRB2": dict(file="AIupred_output_drb2.txt", domain=(189, 434),
                 detrend_win=25, height=0.05, prominence=0.10, distance=6, rel_height=0.50),
    "DRB4": dict(file="AIupred_output_drb4.txt", domain=(151, 291),
                 detrend_win=0,  height=0.55, prominence=0.15, distance=6, rel_height=0.40),
    # DCL4: no PEAK_CFG entry. The ANCHOR2 run for DCL4 (2026-09-02,
    # data/fold_inputs/rna_ds_dcl4_drb2_drb4/AIupred_output_dcl4.txt) is flat
    # across the whole 1529-1620 inter-dsRBD linker -- max 0.41, no
    # MoRF-strength peak -- so there is nothing to carve out. The strong DCL4
    # ANCHOR2 signal (~0.9) is the N-terminal IDR 1-105, which is not a
    # checked region. The DCL4 linker region below is instead trimmed to the
    # residues AIUpred actually calls disordered (see COMPLEXES).
}

def anchor2_morf_residues(cfg):
    a = pd.read_csv(_AIU_DIR / cfg["file"], sep="\t", comment="#", header=None,
                    names=["pos", "aa", "aiupred_disorder", "anchor2"])
    raw, pos = a["anchor2"].to_numpy(), a["pos"].to_numpy()
    sig = raw
    if cfg["detrend_win"]:
        s = pd.Series(raw)
        base = (s.rolling(cfg["detrend_win"], center=True, min_periods=1).min()
                 .rolling(cfg["detrend_win"], center=True, min_periods=1).mean().to_numpy())
        sig = raw - base
    idx, _ = find_peaks(sig, height=cfg["height"], prominence=cfg["prominence"],
                        distance=cfg["distance"])
    _, _, lips, rips = peak_widths(sig, idx, rel_height=cfg["rel_height"])
    to_res = lambda ff: np.interp(ff, np.arange(len(pos)), pos)
    lo, hi = cfg["domain"]
    res = set()
    for k, li, ri in zip(idx, lips, rips):
        if lo <= pos[k] <= hi:
            res |= set(range(max(int(np.floor(to_res(li))), lo),
                             min(int(np.ceil(to_res(ri))), hi) + 1))
    return res

MORF_PEAK_RESIDUES = {prot: anchor2_morf_residues(cfg) for prot, cfg in PEAK_CFG.items()}
_FIXED_WINDOWS = {"DRB2": set(range(425, 435)), "DRB4": set(range(268, 285))}  # old rule, for comparison

def _fmt_runs(residues):
    runs = []
    for r in sorted(residues):
        if runs and r == runs[-1][1] + 1:
            runs[-1][1] = r
        else:
            runs.append([r, r])
    return ",".join(f"{a}" if a == b else f"{a}-{b}" for a, b in runs)

for _p in ("DRB2", "DRB4"):
    print(f"{_p}: {len(MORF_PEAK_RESIDUES[_p])} ANCHOR2-peak MoRF residues "
          f"(fixed-window rule had {len(_FIXED_WINDOWS[_p])})")
    print(f"   ChimeraX: /{'A' if _p == 'DRB2' else 'B'}:{_fmt_runs(MORF_PEAK_RESIDUES[_p])}")

# --- per-complex layout ------------------------------------------------------
COMPLEXES = {
    "rna_ds_drb2_drb4": {
        "chain_names": {"A": "DRB2", "B": "DRB4", "C": "RNA(C)", "D": "RNA(D)"},
        "regions": [
            {"name": "DRB2 disordered", "chain": "A", "start": 189, "end": 434, "morf_protein": "DRB2"},
            {"name": "DRB4 disordered", "chain": "B", "start": 151, "end": 291, "morf_protein": "DRB4"},
        ],
    },
    "rna_ds_dcl4_drb2_drb4": {
        "chain_names": {"A": "DCL4", "B": "DRB2", "C": "DRB4", "D": "RNA(D)", "E": "RNA(E)"},
        "regions": [
            {"name": "DRB2 disordered",         "chain": "B", "start": 189,  "end": 434,  "morf_protein": "DRB2"},
            {"name": "DRB4 disordered",         "chain": "C", "start": 151,  "end": 291,  "morf_protein": "DRB4"},
            # DCL4 inter-dsRBD linker, trimmed to the AIUpred-disordered core.
            # The notebook originally policed the whole structural linker
            # (1529-1620), but the DCL4 AIUpred run
            # (data/fold_inputs/rna_ds_dcl4_drb2_drb4/AIupred_output_dcl4.txt)
            # calls 1529-1585 ORDERED (disorder < 0.2) and only 1586-1623
            # disordered (disorder > 0.5). Helix there is expected, not
            # over-folding, so only 1586-1623 is checked. ANCHOR2 is flat over
            # the window (max 0.41) -> no MoRF carve-out (morf_protein=None).
            {"name": "DCL4 disordered linker core", "chain": "A", "start": 1586, "end": 1623, "morf_protein": None},
        ],
    },
}


DRB2: 132 ANCHOR2-peak MoRF residues (fixed-window rule had 10)
   ChimeraX: /A:199-215,223-236,274-286,290-304,315-328,331-335,342-350,353-361,369-373,380-404,407-412
DRB4: 67 ANCHOR2-peak MoRF residues (fixed-window rule had 17)
   ChimeraX: /B:163-168,187-193,197-208,217-226,243-250,252-256,260-265,272-284


## Filter functions

In [3]:
def read_final_energy(energy_csv_path):
    """Last parseable energy_kJ_mol from a minimize_cif.py *_energy.csv sidecar."""
    if not energy_csv_path.exists():
        return np.nan
    last = np.nan
    with open(energy_csv_path) as fh:
        next(fh, None)  # header
        for line in fh:
            parts = line.strip().split(",")
            if len(parts) < 2:
                continue
            try:
                last = float(parts[1])
            except ValueError:
                pass
    return last


def load_selected(results_dir):
    sel = pd.read_csv(results_dir / "selected_models.csv")
    sel["fname"] = sel["staged_cif"].apply(lambda p: Path(p).stem)
    sel["cluster"] = sel["cluster"].astype(int)
    return sel


def energy_filter(results_dir, sel):
    """Robust MAD z-score on final minimisation energy + whole-backend drop.

    Returns (energy_df, keep_pairs, dropped_backends) where
      energy_df   : one row per minimised model (fname, cluster, backend,
                    final_energy, energy_mod_z, energy_ok, assessed)
      keep_pairs  : set of (cluster, fname) that pass -- (assessed & ok) OR
                    (not assessed), minus every dropped backend
      dropped_backends : backends with >WHOLE_BACKEND_DROP_FRAC flagged
    """
    bl = sel[["fname", "cluster", "backend"]].drop_duplicates()
    rows = []
    for pdb_path in sorted(results_dir.glob("minimized/*/*/*.pdb")):
        if pdb_path.stem.endswith(("_fixed", "_amber", "_nonprot")):
            continue
        cluster = int(pdb_path.parent.parent.name)
        e_csv = pdb_path.with_name(pdb_path.stem + "_energy.csv")
        rows.append({"fname": pdb_path.stem, "cluster": cluster,
                     "final_energy": read_final_energy(e_csv)})
    energy_df = pd.DataFrame(rows).merge(bl, on=["fname", "cluster"], how="left")
    energy_df["assessed"] = energy_df["final_energy"].notna()

    assessed = energy_df[energy_df["assessed"]].copy()
    med = assessed["final_energy"].median()
    mad = (assessed["final_energy"] - med).abs().median()
    energy_df["energy_mod_z"] = 0.6745 * (energy_df["final_energy"] - med) / mad
    energy_df["energy_ok"] = energy_df["energy_mod_z"].abs() <= MOD_Z_THRESHOLD

    flag_rate = (1 - energy_df.loc[energy_df["assessed"]]
                 .groupby("backend")["energy_ok"].mean())
    dropped = sorted(flag_rate[flag_rate > WHOLE_BACKEND_DROP_FRAC].index)

    ok_pairs = set(zip(energy_df.loc[energy_df["assessed"] & energy_df["energy_ok"], "cluster"],
                       energy_df.loc[energy_df["assessed"] & energy_df["energy_ok"], "fname"]))
    unassessed = set(zip(energy_df.loc[~energy_df["assessed"], "cluster"],
                         energy_df.loc[~energy_df["assessed"], "fname"]))
    drop_fnames = set(sel.loc[sel["backend"].isin(dropped), "fname"])
    keep_pairs = {(c, f) for (c, f) in (ok_pairs | unassessed) if f not in drop_fnames}

    energy_df["passes_energy_filter"] = [
        (c, f) in keep_pairs for c, f in zip(energy_df["cluster"], energy_df["fname"])
    ]
    print(f"  energy: pooled median={med:,.0f} kJ/mol, MAD={mad:,.0f}; "
          f"{energy_df['assessed'].sum()}/{len(energy_df)} models energy-assessed")
    print(f"  flagged fraction by backend (|z|>{MOD_Z_THRESHOLD}):")
    print(flag_rate.round(3).to_string().replace("\n", "\n    "))
    print(f"  => whole-backend drop (>{WHOLE_BACKEND_DROP_FRAC:.0%} flagged): {dropped or 'none'}")
    print(f"  => {len(keep_pairs)} / {len(energy_df)} models pass the energy filter")
    return energy_df, keep_pairs, dropped


def helix_runs(dssp_sub):
    """Contiguous helix runs within one (chain, residue-window) slice of dssp.
    Returns a frame of (fname, cluster, run length)."""
    s = dssp_sub.sort_values(["fname", "cluster", "resnum"]).copy()
    s["is_helix"] = s["ss_type"] == "helix"
    grp = s.groupby(["fname", "cluster"], sort=False)["is_helix"]
    s["run_id"] = (s["is_helix"] != grp.shift()).groupby([s["fname"], s["cluster"]]).cumsum()
    ho = s[s["is_helix"]]
    return (ho.groupby(["fname", "cluster", "run_id"])
              .agg(length=("resnum", "size")).reset_index())


def _nonmorf_subranges(lo, hi, morf_residues):
    """Contiguous (a, b) stretches of [lo, hi] after removing every MoRF
    residue -- so a helix run is never counted as bridging an excised gap."""
    subs, start = [], None
    for r in range(lo, hi + 1):
        if r in morf_residues:
            if start is not None:
                subs.append((start, r - 1)); start = None
        elif start is None:
            start = r
    if start is not None:
        subs.append((start, hi))
    return subs


def region_stats(dssp, region, keep_pairs, backend_lookup):
    """Per-model helix stats for one disordered region, gross (full domain) and
    net (ANCHOR2-peak MoRF residues removed). One row per surviving model."""
    chain, lo, hi = region["chain"], region["start"], region["end"]
    morf = MORF_PEAK_RESIDUES.get(region.get("morf_protein"), set())
    base = dssp[(dssp["chain"] == chain) & (dssp["resnum"].between(lo, hi))].copy()
    base = base[[(c, f) in keep_pairs for c, f in zip(base["cluster"], base["fname"])]]

    def _stats(subranges, tag):
        rn = base["resnum"].to_numpy()
        in_any = np.zeros(len(base), dtype=bool)
        for a, b in subranges:
            in_any |= (rn >= a) & (rn <= b)
        sub = base[in_any]
        tot = sub.groupby(["fname", "cluster"]).size().rename(f"n_res_{tag}")
        hel = (sub[sub["ss_type"] == "helix"].groupby(["fname", "cluster"]).size()
               .rename(f"n_helix_{tag}"))
        mx = {}
        for a, b in subranges:                       # runs per contiguous sub-range
            rr = helix_runs(sub[sub["resnum"].between(a, b)])
            for (fn, cl), g in rr.groupby(["fname", "cluster"]):
                mx[(fn, cl)] = max(mx.get((fn, cl), 0), int(g["length"].max()))
        out = pd.concat([tot, hel], axis=1)
        out[f"n_helix_{tag}"] = out[f"n_helix_{tag}"].fillna(0)
        out[f"max_helix_run_{tag}"] = [mx.get(k, 0) for k in out.index]
        out[f"pct_helix_{tag}"] = 100 * out[f"n_helix_{tag}"] / out[f"n_res_{tag}"]
        return out

    gross = _stats([(lo, hi)], "gross")
    net = _stats(_nonmorf_subranges(lo, hi, morf), "net")
    df = gross.join(net, how="outer").reset_index()
    df["region"] = region["name"]
    df["chain"] = chain
    df = df.merge(backend_lookup, on=["fname", "cluster"], how="left")
    df["overfolded"] = (df["pct_helix_net"] > MAX_PCT_HELIX_NONMORF) | \
                       (df["max_helix_run_net"] >= MIN_LONG_HELIX_RUN)
    return df


def run_complex(name, drop_regions=(), variant=""):
    """Energy + non-MoRF over-folding filter for one complex.

    drop_regions : region names (matching COMPLEXES[name]["regions"][*]["name"])
                   to leave *out* of the over-folding filter entirely -- not
                   policed, not reported, not exported. Used for the
                   "DCL4 over-folding filter disabled" variant, where the DCL4
                   disordered linker core is dropped so only the DRB2 / DRB4
                   regions decide whether a model is kept.
    variant      : output-folder suffix. Carried in the returned dict and
                   threaded through every save_fig / out_path / export call so
                   this run's figures and filtered_models.csv go to
                   `energy_overfolding_filter_<variant>/` instead of
                   overwriting the canonical run's outputs.
    """
    cfg = COMPLEXES[name]
    drop_regions = set(drop_regions)
    regions = [r for r in cfg["regions"] if r["name"] not in drop_regions]
    results_dir = ROOT / "results" / name
    print(f"=== {name}{f'  [variant: {variant}]' if variant else ''} ===")
    if drop_regions:
        print(f"  over-folding filter: DROPPED region(s) {sorted(drop_regions)}")
        print(f"  over-folding filter: checking {[r['name'] for r in regions]}")
    sel = load_selected(results_dir)
    backend_lookup = sel[["fname", "cluster", "backend"]].drop_duplicates()

    energy_df, keep_pairs, dropped = energy_filter(results_dir, sel)

    dssp = pd.read_csv(results_dir / "dssp_summary.csv")
    dssp["cluster"] = dssp["cluster"].astype(int)

    region_frames = [region_stats(dssp, r, keep_pairs, backend_lookup) for r in regions]
    per_region = pd.concat(region_frames, ignore_index=True)

    # collapse to one row per model: over-folded if ANY checked region is
    model = (per_region.groupby(["fname", "cluster", "backend"])["overfolded"]
             .any().rename("overfolded_any_region").reset_index())
    model["passes_overfolding_filter"] = ~model["overfolded_any_region"]
    model["keep"] = model["passes_overfolding_filter"]  # (already energy-filtered set)

    e_slim = energy_df[["fname", "cluster", "final_energy", "energy_mod_z"]]
    model = model.merge(e_slim, on=["fname", "cluster"], how="left")

    n_energy = len(keep_pairs)
    n_both = int(model["keep"].sum())
    print(f"  over-folding: {n_both} / {n_energy} energy-survivors also pass "
          f"(pct_helix_net > {MAX_PCT_HELIX_NONMORF} OR max_helix_run_net >= {MIN_LONG_HELIX_RUN} "
          f"in any checked region => dropped)\n")

    return {"name": name, "cfg": cfg, "regions": regions, "variant": variant,
            "sel": sel, "energy_df": energy_df,
            "keep_pairs": keep_pairs, "dropped_backends": dropped,
            "per_region": per_region, "model": model,
            "n_energy": n_energy, "n_both": n_both}

## Plot / report helpers

In [4]:
def report_survival(res):
    """Per-backend funnel: total selected -> energy-filtered -> +over-folding."""
    name, variant = res["name"], res.get("variant", "")
    sel, energy_df, model = res["sel"], res["energy_df"], res["model"]
    total = sel.groupby("backend")["fname"].nunique().rename("selected")
    e_ok = (energy_df[energy_df["passes_energy_filter"]]
            .groupby("backend")["fname"].nunique().rename("after_energy_filter"))
    both = model[model["keep"]].groupby("backend")["fname"].nunique().rename("after_both_filters")
    tab = pd.concat([total, e_ok, both], axis=1).fillna(0).astype(int)
    tab.loc["TOTAL"] = tab.sum()
    print(f"[{name}] models surviving each filter, by backend:")
    display(tab)

    plot = tab.drop(index="TOTAL").reset_index().melt(
        id_vars="backend", var_name="stage", value_name="n_models")
    stage_order = ["selected", "after_energy_filter", "after_both_filters"]
    fig = px.bar(plot, x="backend", y="n_models", color="stage", barmode="group",
                 category_orders={"stage": stage_order},
                 color_discrete_sequence=px.colors.qualitative.Safe, template=TEMPLATE,
                 title=f"{name}: models surviving the energy and non-MoRF over-folding filters")
    fig.update_layout(width=820, height=460, xaxis_title="", yaxis_title="models")
    save_fig(fig, name, "survival_by_backend.html", variant)
    return tab


def plot_energy(res):
    name, variant = res["name"], res.get("variant", "")
    e = res["energy_df"].dropna(subset=["final_energy"]).copy()
    e["flag"] = np.where(e["passes_energy_filter"], "kept", "garbage energy / dropped backend")
    # energies span negative (well-minimised) to ~1e19 (divergent) -- a plain
    # linear or log axis can't show both. symlog: sign(x) * log10(1 + |x|).
    e["energy_symlog"] = np.sign(e["final_energy"]) * np.log10(1 + e["final_energy"].abs())
    order = e.groupby("backend")["final_energy"].median().sort_values().index.tolist()
    fig = px.strip(e, x="backend", y="energy_symlog", color="flag",
                   category_orders={"backend": order},
                   color_discrete_map={"kept": "#4C72B0",
                                       "garbage energy / dropped backend": "#C44E52"},
                   hover_data={"final_energy": ":.1f", "energy_mod_z": ":.2f", "energy_symlog": False},
                   template=TEMPLATE,
                   title=f"{name}: final minimisation energy per model  [y = sign·log10(1+|kJ/mol|)]")
    fig.update_traces(jitter=0.35, marker=dict(size=5, opacity=0.7))
    fig.update_layout(width=820, height=460, xaxis_title="",
                      yaxis_title="final energy  (symlog kJ/mol)", legend_title="")
    print(f"[{name}] final energy by backend (kJ/mol):")
    display(e.groupby("backend")["final_energy"].agg(["count", "min", "median", "max"]).round(1))
    save_fig(fig, name, "final_energy_by_backend.html", variant)


def plot_overfolding(res):
    """Per-backend distribution of the two over-folding metrics in the non-MoRF
    disordered residues, faceted by region, with the threshold line drawn in."""
    name, variant = res["name"], res.get("variant", "")
    pr = res["per_region"].copy()
    region_order = [r["name"] for r in res.get("regions", res["cfg"]["regions"])]

    for metric, thresh, thr_kind, ytitle in [
        ("pct_helix_net", MAX_PCT_HELIX_NONMORF, ">", "% helix (non-MoRF residues)"),
        ("max_helix_run_net", MIN_LONG_HELIX_RUN, ">=", "longest helix run (residues, non-MoRF)"),
    ]:
        fig = px.strip(pr, x="backend", y=metric, color="backend", facet_col="region",
                       category_orders={"region": region_order},
                       color_discrete_sequence=BACKEND_PALETTE, template=TEMPLATE,
                       title=f"{name}: {ytitle} — dashed line = filter cutoff ({thr_kind} {thresh} → over-folded)")
        fig.update_traces(jitter=0.4, marker=dict(size=5, opacity=0.65))
        fig.add_hline(y=thresh, line_dash="dash", line_color="black", opacity=0.7)
        fig.update_layout(width=340 * len(region_order) + 120, height=460, showlegend=False,
                          xaxis_title="")
        fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
        save_fig(fig, name, f"overfolding_{metric}_by_backend.html", variant)

    # gross vs net: does carving out the MoRF windows change the picture at all?
    g = (pr.groupby(["region", "backend"])[["pct_helix_gross", "pct_helix_net"]]
         .median().reset_index()
         .melt(id_vars=["region", "backend"], var_name="which", value_name="pct_helix"))
    g["which"] = g["which"].map({"pct_helix_gross": "gross (full domain)",
                                 "pct_helix_net": "net (MoRF window removed)"})
    fig = px.bar(g, x="backend", y="pct_helix", color="which", barmode="group",
                 facet_col="region", category_orders={"region": region_order},
                 color_discrete_sequence=["#C44E52", "#4C72B0"], template=TEMPLATE,
                 title=f"{name}: median % helix — gross vs. MoRF-excluded (net)")
    fig.update_layout(width=340 * len(region_order) + 120, height=430, xaxis_title="",
                      legend_title="")
    fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
    save_fig(fig, name, "overfolding_gross_vs_net.html", variant)


def export_survivors(res):
    """Write the surviving (cluster, fname, backend) + per-region stats to CSV."""
    name, variant = res["name"], res.get("variant", "")
    wide = res["per_region"].pivot_table(
        index=["fname", "cluster", "backend"],
        columns="region",
        values=["pct_helix_net", "max_helix_run_net", "overfolded"],
    )
    wide.columns = [f"{a}::{b}" for a, b in wide.columns]
    wide = wide.reset_index()
    out = res["model"][["fname", "cluster", "backend", "final_energy", "energy_mod_z",
                        "passes_overfolding_filter", "keep"]].merge(
        wide, on=["fname", "cluster", "backend"], how="left")
    out = out.sort_values(["keep", "backend", "fname"], ascending=[False, True, True])
    p = out_path(name, "filtered_models.csv", variant)
    out.to_csv(p, index=False)
    print(f"Saved: {p}  ({int(out['keep'].sum())} kept / {len(out)} energy-survivors)")
    return out


def threshold_sweep(res, pcts=(15, 20, 25, 30, 40), runs=(10, 12, 15, 20)):
    pr = res["per_region"]
    grid = []
    for pct in pcts:
        row = {"pct_helix_net >": pct}
        for run in runs:
            bad = pr[(pr["pct_helix_net"] > pct) | (pr["max_helix_run_net"] >= run)]
            bad_keys = set(zip(bad["fname"], bad["cluster"]))
            row[f"run>={run}"] = res["n_energy"] - len(bad_keys)
        grid.append(row)
    g = pd.DataFrame(grid).set_index("pct_helix_net >")
    print(f"[{res['name']}] models surviving BOTH filters (of {res['n_energy']} energy-survivors), "
          f"over a threshold grid:")
    display(g)
    return g

---
## Complex 1 — `rna_ds_drb2_drb4` (DRB2 / DRB4 / dsRNA, no DCL4)

Six backends were submitted; RosettaFold3's minimisations diverge numerically
and it is dropped by the energy filter, leaving AlphaFold3 / Boltz / Chai-1 /
OpenFold3 / Protenix.

In [5]:
res1 = run_complex('rna_ds_drb2_drb4')

=== rna_ds_drb2_drb4 ===


  energy: pooled median=3,838 kJ/mol, MAD=4,803; 539/598 models energy-assessed
  flagged fraction by backend (|z|>3.5):
backend
    alphafold3      0.000
    boltz           0.000
    chai1           0.011
    openfold3       0.000
    protenix        0.000
    rosettafold3    0.833
  => whole-backend drop (>50% flagged): ['rosettafold3']
  => 499 / 598 models pass the energy filter


  over-folding: 200 / 499 energy-survivors also pass (pct_helix_net > 20.0 OR max_helix_run_net >= 12 in any checked region => dropped)



In [6]:
plot_energy(res1)

[rna_ds_drb2_drb4] final energy by backend (kJ/mol):


,count,min,median,max
backend,,,,
alphafold3,99,-4718.8,9.674500e+03,2.791940e+04
boltz,94,-9108.5,1.107600e+03,1.495010e+04
chai1,90,-6972.7,1.853200e+03,3.335460e+04
openfold3,99,-6267.3,3.326000e+03,1.836620e+04
protenix,91,-11008.4,8.352000e+02,1.620430e+04
rosettafold3,66,-6265.0,2.308130e+09,8.537021e+18


Saved: ../results/rna_ds_drb2_drb4/figures/domain_analysis/energy_overfolding_filter/final_energy_by_backend.html


In [7]:
plot_overfolding(res1)

Saved: ../results/rna_ds_drb2_drb4/figures/domain_analysis/energy_overfolding_filter/overfolding_pct_helix_net_by_backend.html


Saved: ../results/rna_ds_drb2_drb4/figures/domain_analysis/energy_overfolding_filter/overfolding_max_helix_run_net_by_backend.html


Saved: ../results/rna_ds_drb2_drb4/figures/domain_analysis/energy_overfolding_filter/overfolding_gross_vs_net.html


In [8]:
tab1 = report_survival(res1)

[rna_ds_drb2_drb4] models surviving each filter, by backend:


,selected,after_energy_filter,after_both_filters
backend,,,
alphafold3,100,100,82
boltz,100,100,5
chai1,100,99,50
openfold3,100,100,0
protenix,100,100,63
rosettafold3,100,0,0
TOTAL,600,499,200


Saved: ../results/rna_ds_drb2_drb4/figures/domain_analysis/energy_overfolding_filter/survival_by_backend.html


In [9]:
sweep1 = threshold_sweep(res1)

[rna_ds_drb2_drb4] models surviving BOTH filters (of 499 energy-survivors), over a threshold grid:


,run>=10,run>=12,run>=15,run>=20
pct_helix_net >,,,,
15,149,154,157,157
20,185,200,205,205
25,205,241,252,254
30,225,279,301,311
40,238,298,336,358


In [10]:
out1 = export_survivors(res1)

Saved: ../results/rna_ds_drb2_drb4/figures/domain_analysis/energy_overfolding_filter/filtered_models.csv  (200 kept / 499 energy-survivors)


### Reading complex 1

- **Energy filter** removes RosettaFold3 wholesale (five backends, 499 models
  go forward).
- **Non-MoRF over-folding filter** — with the ANCHOR2-peak MoRF definition,
  **200 / 499** models survive both filters:

  | backend | after energy | after both | (fixed-window rule) |
  |---|---|---|---|
  | alphafold3 | 100 | 82 | 83 |
  | protenix | 100 | 63 | 77 |
  | chai1 | 99 | 50 | 37 |
  | boltz | 100 | 5 | 7 |
  | openfold3 | 100 | 0 | 0 |

- **Switching from the two fixed windows (DRB2 425–434 / DRB4 268–284, 27
  residues total) to the ANCHOR2-peak regions (~199 residues, ~13× more
  exempt) barely moves the total: 196 → 200.** It only reshuffles which
  backends pass — Chai-1 gains (37 → 50, its helix was concentrated in the
  now-exempt mid-tail), Protenix loses (77 → 63, its low-level helix sits at
  the disordered-domain *start*, 189–~224, which the ANCHOR2 peaks do **not**
  cover). OpenFold3 stays wiped either way (~55–60 % helix across the whole
  domain, exempt regions included).
- **Why so little effect:** the over-folding in flagged models is broad —
  smeared across the full disordered domain, including the residues *between*
  ANCHOR2 peaks and the stretch before the first peak — so exempting the
  peak regions doesn't rescue a model that is also helical everywhere else.
- **What's left** is still an AlphaFold3 + Protenix + (now) Chai-1 subset —
  see `filtered_models.csv`.

---
## Complex 2 — `rna_ds_dcl4_drb2_drb4` (DCL4 / DRB2 / DRB4 / dsRNA — the full complex)

Only AlphaFold3, OpenFold3 and RosettaFold3 produced models here (Boltz /
Chai-1 / Protenix hard-failed this complex's ~2600-token input — see
`rna_ds_dcl4_drb2_drb4_domain_analysis.ipynb`). A third disordered region is
checked here: the **AIUpred-disordered core of DCL4's inter-dsRBD linker,
1586–1623**. The DCL4 ANCHOR2 run (2026-09-02,
`data/fold_inputs/rna_ds_dcl4_drb2_drb4/AIupred_output_dcl4.txt`) is flat
across the whole structural linker (1529–1620, max ANCHOR2 0.41 — no
MoRF-strength peak), and AIUpred calls 1529–1585 ordered (disorder < 0.2),
so the region is trimmed to the genuine IDR with no MoRF carve-out. (It was
previously the whole 1529–1620 linker, "fully policed" — which wrongly
flagged models that build helix in the ordered 1529–1585 stretch.)

In [11]:
res2 = run_complex('rna_ds_dcl4_drb2_drb4')

=== rna_ds_dcl4_drb2_drb4 ===


  energy: pooled median=-155,632 kJ/mol, MAD=6,250; 241/297 models energy-assessed
  flagged fraction by backend (|z|>3.5):
backend
    alphafold3      0.045
    openfold3       0.031
    rosettafold3    1.000
  => whole-backend drop (>50% flagged): ['rosettafold3']
  => 193 / 297 models pass the energy filter


  over-folding: 1 / 193 energy-survivors also pass (pct_helix_net > 20.0 OR max_helix_run_net >= 12 in any checked region => dropped)



In [12]:
plot_energy(res2)

[rna_ds_dcl4_drb2_drb4] final energy by backend (kJ/mol):


,count,min,median,max
backend,,,,
alphafold3,88,-166934.1,-1.560310e+05,1.580500e+04
openfold3,97,-169323.1,-1.599933e+05,-1.570620e+04
rosettafold3,56,946478.8,1.393822e+11,6.195695e+16


Saved: ../results/rna_ds_dcl4_drb2_drb4/figures/domain_analysis/energy_overfolding_filter/final_energy_by_backend.html


In [13]:
plot_overfolding(res2)

Saved: ../results/rna_ds_dcl4_drb2_drb4/figures/domain_analysis/energy_overfolding_filter/overfolding_pct_helix_net_by_backend.html


Saved: ../results/rna_ds_dcl4_drb2_drb4/figures/domain_analysis/energy_overfolding_filter/overfolding_max_helix_run_net_by_backend.html


Saved: ../results/rna_ds_dcl4_drb2_drb4/figures/domain_analysis/energy_overfolding_filter/overfolding_gross_vs_net.html


In [14]:
tab2 = report_survival(res2)

[rna_ds_dcl4_drb2_drb4] models surviving each filter, by backend:


,selected,after_energy_filter,after_both_filters
backend,,,
alphafold3,100,96,1
openfold3,100,97,0
rosettafold3,100,0,0
TOTAL,300,193,1


Saved: ../results/rna_ds_dcl4_drb2_drb4/figures/domain_analysis/energy_overfolding_filter/survival_by_backend.html


In [15]:
sweep2 = threshold_sweep(res2)

[rna_ds_dcl4_drb2_drb4] models surviving BOTH filters (of 193 energy-survivors), over a threshold grid:


,run>=10,run>=12,run>=15,run>=20
pct_helix_net >,,,,
15,0,0,0,0
20,1,1,1,1
25,1,1,1,1
30,1,1,1,1
40,2,2,2,2


In [16]:
out2 = export_survivors(res2)

Saved: ../results/rna_ds_dcl4_drb2_drb4/figures/domain_analysis/energy_overfolding_filter/filtered_models.csv  (1 kept / 193 energy-survivors)


---
## Symlink the survivors for ChimeraX inspection

`scripts/symlink_filtered_survivors.py` projects each complex's
`filtered_models.csv` (written just above by `export_survivors`) into a flat
per-backend tree of symlinks to the minimised `.pdb`s, cleanest first:

```
results/<complex>/overfolding_inspection_survivors/<backend>/NN_maxhelix<run>_pct<pct>_<fname>.pdb
results/<complex>/overfolding_inspection_survivors/_survivors.tsv     # manifest
```

Re-running this notebook (new AIUpred/ANCHOR2 data, a threshold change, a
region-boundary change) rewrites `filtered_models.csv` and then this cell
rebuilds the tree from scratch — newly-passing models appear, models that no
longer pass are dropped. Add `--also-worst 20` to the call to additionally
populate `overfolding_inspection/<backend>/` with the worst offenders.

(For `rna_ds_drb2_drb4` this shares the directory name with
`rna_ds_drb2_drb4_domain_analysis.ipynb`'s own `_write_symlinks` output —
same idea, near-identical filter — so whichever ran last wins. Pass
`--out-dir` if you need them side by side.)

In [17]:
import subprocess, sys

for _cx in COMPLEXES:
    print(f"\n=== {_cx} ===")
    subprocess.run(
        [sys.executable, str(ROOT / "scripts" / "symlink_filtered_survivors.py"),
         "--complex", _cx, "--results-root", str(ROOT / "results")],
        check=True,
    )


=== rna_ds_drb2_drb4 ===


[rna_ds_drb2_drb4] 200 survivor(s) / 499 energy-survivors  (regions: DRB2 disordered, DRB4 disordered)
[survivors] -> ../results/rna_ds_drb2_drb4/overfolding_inspection_survivors
  alphafold3  :  82 linked  (of 82 in group)
  boltz       :   5 linked  (of 5 in group)
  chai1       :  50 linked  (of 50 in group)
  protenix    :  63 linked  (of 63 in group)
[survivors] manifest -> ../results/rna_ds_drb2_drb4/overfolding_inspection_survivors/_survivors.tsv

Done: 200 survivor symlink(s) under ../results/rna_ds_drb2_drb4/overfolding_inspection_survivors

=== rna_ds_dcl4_drb2_drb4 ===


[rna_ds_dcl4_drb2_drb4] 1 survivor(s) / 193 energy-survivors  (regions: DCL4 disordered linker core, DRB2 disordered, DRB4 disordered)
[survivors] -> ../results/rna_ds_dcl4_drb2_drb4/overfolding_inspection_survivors
  alphafold3  :   1 linked  (of 1 in group)
[survivors] manifest -> ../results/rna_ds_dcl4_drb2_drb4/overfolding_inspection_survivors/_survivors.tsv

Done: 1 survivor symlink(s) under ../results/rna_ds_dcl4_drb2_drb4/overfolding_inspection_survivors


### Reading complex 2 — the over-folding is near-total, MoRF definition irrelevant

- **Energy filter** removes RosettaFold3, leaving AlphaFold3 + OpenFold3
  (193 models).
- **Non-MoRF over-folding filter leaves just one — `1 / 193`.** With DCL4 in
  the complex, AlphaFold3 over-folds the DRB2/DRB4 disordered regions almost
  as badly as OpenFold3: across the 193 energy-survivors the *non-MoRF*
  median % helix is ~62 % (DRB2, over-folded in 191/193), ~57 % (DRB4,
  155/193), with longest runs up to 34/18. Exempting the ~199 ANCHOR2-peak
  residues instead of the 27 fixed-window ones still changes the survivor
  count by **0** on the DRB2/DRB4 side: the helix content is high across the
  whole domain, peak regions and gaps alike.
- **DCL4 disordered linker core (1586–1623), retrimmed from the old 1529–1620
  "fully policed" linker after the 2026-09-02 DCL4 AIUpred run.** 1529–1585
  is predicted ordered (helix there is expected, not hallucination) and was
  wrongly counted before; the genuine IDR 1586–1623 is still heavily policed
  — over-folded in **153/193**, median ~45 % helix, longest runs up to 27 —
  so trimming does not "let DCL4 off". It only stops penalising helix in the
  ordered flank.
- **The one survivor is `rank_09_alphafold3_seed10_sample2.0` (cluster 1):**
  DRB2 3.5 % / DRB4 0 % / DCL4-core 15.8 % helix, longest non-MoRF runs
  4 / 0 / 3 — clean on all three regions. Under the old DCL4 boundary it was
  the model flagged only by helices at 1536–1543 & 1565–1581, i.e. in the
  predicted-ordered part of the linker; correcting the boundary is exactly
  what recovers it. The threshold sweep now reads 1 at `pct>20`, 2 at
  `pct>40`.
- **Conclusion essentially unchanged:** there is no *ensemble* of
  trustworthy disordered-region models for the full DCL4 complex — 1/193 is
  not a usable set. The folded cores (DCL4 domains, dsRBDs, RNA) are
  unaffected by this filter.

---
## Complex 2 variant — `rna_ds_dcl4_drb2_drb4` with the DCL4 over-folding filter **disabled**

Identical to the run just above (same energy filter, same DRB2 / DRB4
non-MoRF over-folding filter) **except the DCL4 disordered linker core
(1586–1623) is not policed at all**. It is dropped from the filter
(`drop_regions=("DCL4 disordered linker core",)`), so only the DRB2 and DRB4
disordered regions decide whether a model survives the over-folding filter.

Why: the DCL4 linker carries no ANCHOR2 MoRF signal to exempt and its AIUpred
disorder call is softer than the DRBs' — downstream steps that only need
clean DRB2 / DRB4 tails were being blocked by DCL4-linker helix. This run
serves them without disturbing the canonical result above.

All outputs carry the `no_dcl4` suffix and land in **sibling folders**, so
nothing above is overwritten:

```
results/rna_ds_dcl4_drb2_drb4/figures/domain_analysis/energy_overfolding_filter_no_dcl4/
    ├── final_energy_by_backend.html
    ├── overfolding_*_by_backend.html
    ├── survival_by_backend.html
    └── filtered_models.csv
results/rna_ds_dcl4_drb2_drb4/overfolding_inspection_survivors_no_dcl4/
```

In [18]:
res2_no_dcl4 = run_complex(
    "rna_ds_dcl4_drb2_drb4",
    drop_regions=("DCL4 disordered linker core",),
    variant="no_dcl4",
)

=== rna_ds_dcl4_drb2_drb4  [variant: no_dcl4] ===
  over-folding filter: DROPPED region(s) ['DCL4 disordered linker core']
  over-folding filter: checking ['DRB2 disordered', 'DRB4 disordered']
  energy: pooled median=-155,632 kJ/mol, MAD=6,250; 241/297 models energy-assessed
  flagged fraction by backend (|z|>3.5):
backend
    alphafold3      0.045
    openfold3       0.031
    rosettafold3    1.000
  => whole-backend drop (>50% flagged): ['rosettafold3']
  => 193 / 297 models pass the energy filter


  over-folding: 1 / 193 energy-survivors also pass (pct_helix_net > 20.0 OR max_helix_run_net >= 12 in any checked region => dropped)



In [19]:
plot_energy(res2_no_dcl4)

[rna_ds_dcl4_drb2_drb4] final energy by backend (kJ/mol):


,count,min,median,max
backend,,,,
alphafold3,88,-166934.1,-1.560310e+05,1.580500e+04
openfold3,97,-169323.1,-1.599933e+05,-1.570620e+04
rosettafold3,56,946478.8,1.393822e+11,6.195695e+16


Saved: ../results/rna_ds_dcl4_drb2_drb4/figures/domain_analysis/energy_overfolding_filter_no_dcl4/final_energy_by_backend.html


In [20]:
plot_overfolding(res2_no_dcl4)

Saved: ../results/rna_ds_dcl4_drb2_drb4/figures/domain_analysis/energy_overfolding_filter_no_dcl4/overfolding_pct_helix_net_by_backend.html


Saved: ../results/rna_ds_dcl4_drb2_drb4/figures/domain_analysis/energy_overfolding_filter_no_dcl4/overfolding_max_helix_run_net_by_backend.html


Saved: ../results/rna_ds_dcl4_drb2_drb4/figures/domain_analysis/energy_overfolding_filter_no_dcl4/overfolding_gross_vs_net.html


In [21]:
tab2_no_dcl4 = report_survival(res2_no_dcl4)

[rna_ds_dcl4_drb2_drb4] models surviving each filter, by backend:


,selected,after_energy_filter,after_both_filters
backend,,,
alphafold3,100,96,1
openfold3,100,97,0
rosettafold3,100,0,0
TOTAL,300,193,1


Saved: ../results/rna_ds_dcl4_drb2_drb4/figures/domain_analysis/energy_overfolding_filter_no_dcl4/survival_by_backend.html


In [22]:
sweep2_no_dcl4 = threshold_sweep(res2_no_dcl4)

[rna_ds_dcl4_drb2_drb4] models surviving BOTH filters (of 193 energy-survivors), over a threshold grid:


,run>=10,run>=12,run>=15,run>=20
pct_helix_net >,,,,
15,1,1,1,1
20,1,1,1,1
25,1,1,1,1
30,1,1,1,1
40,2,2,2,2


In [23]:
out2_no_dcl4 = export_survivors(res2_no_dcl4)

Saved: ../results/rna_ds_dcl4_drb2_drb4/figures/domain_analysis/energy_overfolding_filter_no_dcl4/filtered_models.csv  (1 kept / 193 energy-survivors)


### Symlink the DCL4-filter-disabled survivors

Same projection as the canonical symlink cell, pointed at this variant's
`filtered_models.csv` and writing to `overfolding_inspection_survivors_no_dcl4/`
so it never collides with the canonical `overfolding_inspection_survivors/`.

In [24]:
import subprocess, sys

_variant_csv = out_path("rna_ds_dcl4_drb2_drb4", "filtered_models.csv", "no_dcl4")
_variant_out = (ROOT / "results" / "rna_ds_dcl4_drb2_drb4"
                / "overfolding_inspection_survivors_no_dcl4")
subprocess.run(
    [sys.executable, str(ROOT / "scripts" / "symlink_filtered_survivors.py"),
     "--complex", "rna_ds_dcl4_drb2_drb4", "--results-root", str(ROOT / "results"),
     "--filtered-csv", str(_variant_csv), "--out-dir", str(_variant_out)],
    check=True,
)

[rna_ds_dcl4_drb2_drb4] 1 survivor(s) / 193 energy-survivors  (regions: DRB2 disordered, DRB4 disordered)
[survivors] -> ../results/rna_ds_dcl4_drb2_drb4/overfolding_inspection_survivors_no_dcl4
  alphafold3  :   1 linked  (of 1 in group)
[survivors] manifest -> ../results/rna_ds_dcl4_drb2_drb4/overfolding_inspection_survivors_no_dcl4/_survivors.tsv

Done: 1 survivor symlink(s) under ../results/rna_ds_dcl4_drb2_drb4/overfolding_inspection_survivors_no_dcl4


CompletedProcess(args=['/opt/homebrew/Cellar/micromamba/2.8.1/envs/abcfold-drbs-notebook/bin/python', '../scripts/symlink_filtered_survivors.py', '--complex', 'rna_ds_dcl4_drb2_drb4', '--results-root', '../results', '--filtered-csv', '../results/rna_ds_dcl4_drb2_drb4/figures/domain_analysis/energy_overfolding_filter_no_dcl4/filtered_models.csv', '--out-dir', '../results/rna_ds_dcl4_drb2_drb4/overfolding_inspection_survivors_no_dcl4'], returncode=0)

### Reading — DCL4 filter disabled

Compare `tab2_no_dcl4` / `sweep2_no_dcl4` here against `tab2` / `sweep2`
above. The difference is exactly the set of models that passed DRB2 **and**
DRB4 but were being dropped *only* for DCL4-linker-core helix — the DRB2 /
DRB4 verdicts are unchanged (same regions, same thresholds).

- If the survivor count is still ~0, the DRB2 / DRB4 tails are over-folded on
  their own and disabling the DCL4 check does not recover a usable ensemble.
- If it jumps, DCL4-linker helix was the sole blocker for those models and
  this variant's `filtered_models.csv` /
  `overfolding_inspection_survivors_no_dcl4/` is the set to inspect.

---
## Summary

| | `rna_ds_drb2_drb4` | `rna_ds_dcl4_drb2_drb4` |
|---|---|---|
| backends after energy filter | AF3, Boltz, Chai-1, OpenFold3, Protenix | AF3, OpenFold3 |
| dropped by energy filter | RosettaFold3 (divergent) | RosettaFold3 (divergent) |
| survive over-folding filter — **fixed MoRF windows** (27 res) | 196 / 499 | 0 / 193 |
| survive over-folding filter — **ANCHOR2-peak MoRF** (~199 res) | **200 / 499** | **1 / 193** |
| survive over-folding filter — **DCL4 region disabled** (`no_dcl4` variant) | n/a | **1 / 193** |
| interpretation | usable AF3/Protenix/Chai-1 subset | disordered regions unreliable across all backends |

**The ANCHOR2-peak MoRF definition makes essentially no difference to the
outcome** — +4 models for the DCL4-free complex (a reshuffle: Chai-1 up,
Protenix down), and for the full DCL4 complex the only movement is +1 from
**retrimming the DCL4 region** (old 1529–1620 structural linker → 1586–1623
AIUpred-disordered core, after the 2026-09-02 DCL4 AIUpred/ANCHOR2 run), not
from the MoRF exemption itself — ANCHOR2 is flat over the whole DCL4 linker.
The reason the exemption barely matters is the same in both complexes: the
disordered-region over-folding this filter catches is broad, not localised to
(or away from) the ANCHOR2 crests, so how the MoRF exemption is drawn hardly
changes anything. The user's expectation held: worth testing, no real
improvement.

The single `rna_ds_dcl4_drb2_drb4` survivor
(`rank_09_alphafold3_seed10_sample2.0`) was recovered specifically by the
DCL4 boundary correction: its only helices in the old 1529–1620 window sat
in the predicted-**ordered** flank 1529–1585, where secondary structure is
expected.

**DCL4-over-folding-filter-disabled variant** (`run_complex(..., drop_regions=("DCL4
disordered linker core",), variant="no_dcl4")` — outputs under
`figures/domain_analysis/energy_overfolding_filter_no_dcl4/` and
`overfolding_inspection_survivors_no_dcl4/`): checking only DRB2 / DRB4
leaves **1 / 193**, the *same* single AF3 model. The DCL4 linker core was
therefore **not** the blocker — the DRB2 / DRB4 tails are over-folded on their
own in 192 / 193 energy-survivors. Disabling the DCL4 check does not open up
a usable ensemble for the full complex.

Tune `MAX_PCT_HELIX_NONMORF` / `MIN_LONG_HELIX_RUN` at the top (or the
`PEAK_CFG` detection params) and re-run — the sweep tables show complex 1 is
moderately threshold-sensitive, complex 2 stays at 0–2 regardless.